In [ ]:
# =========================
# SAMLET PIPELINE + CONSISTENT CUT-OFF + AUDIT
# Unikke bygninger, matrikelklasser og smitterisiko
# =========================

from __future__ import annotations

from pathlib import Path
import zipfile
import hashlib
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

KOMMUNE_KODE = "0101"

# Juridisk/datamæssig cut-off:
# "taget i brug efter 31/12/1991" -> proxy: opført fra og med 1992
OLD_MAX_YEAR = 1991
NEW_MIN_YEAR = 1992
CUT_YEAR = NEW_MIN_YEAR

MIN_PLAUSIBLE_YEAR = 1500
MAX_PLAUSIBLE_YEAR = 2026

YEAR_COL = "byg026Opførelsesår"

DATA = Path("data")
EXPORT = Path("exports")
DATA.mkdir(exist_ok=True)
EXPORT.mkdir(exist_ok=True)

# Inputfiler
BYG_ZIP = DATA / "BBR_V3_Bygning_0101_TotalDownload_csv_Current_535.zip"
GJ_ZIP  = DATA / "BBR_V3_GrundJordstykke_0101_TotalDownload_csv_Current_535.zip"
MAT_DIR = DATA / "mat_jordstykke"

# Låst efter din audit
MAT_KEY_COL = "id_lokalId"

# Sæt til True, hvis du også vil eksportere hele building-level datasættet
EXPORT_VALID_BUILDING_LEVEL = False


# ============================================================
# HELPERS
# ============================================================

def unzip_largest_csv(zip_path: Path, out_dir: Path) -> Path:
    """
    Udpakker zip og returnerer den største CSV.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    csvs = list(out_dir.rglob("*.csv"))
    if not csvs:
        raise RuntimeError(f"Fandt ingen CSV i {zip_path}")

    csvs.sort(key=lambda p: p.stat().st_size, reverse=True)
    return csvs[0]


def sha256(path: Path, chunk: int = 1024 * 1024) -> str:
    """
    Laver SHA256-hash til input-provenance.
    """
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)
    return h.hexdigest()


def norm_str(s: pd.Series) -> pd.Series:
    """
    Normaliserer tekstkolonner deterministisk.
    """
    return s.astype("string").str.strip()


def to_int(s: pd.Series) -> pd.Series:
    """
    Konverterer til nullable integer.
    """
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def norm_join_key(s: pd.Series) -> pd.Series:
    """
    Robust join-key til jordstykke-id'er.

    Formål:
    - Hvis værdien er numerisk, normaliseres den til integer-string.
    - Hvis værdien ikke kan læses som tal, beholdes den som trimmed string.
    - Tomme værdier sættes til NA.

    Det gør joinet mere robust end ren to_int(), men bevarer stadig
    kompatibilitet med numeriske id'er.
    """
    raw = norm_str(s)

    empty_mask = (
        raw.isna()
        | raw.eq("")
        | raw.str.lower().isin(["nan", "none", "<na>"])
    )

    raw_clean = raw.str.replace(r"\.0$", "", regex=True)
    numeric = pd.to_numeric(raw_clean, errors="coerce")

    out = raw_clean.copy()
    numeric_mask = numeric.notna()

    out.loc[numeric_mask] = numeric.loc[numeric_mask].astype("Int64").astype("string")
    out.loc[empty_mask] = pd.NA

    return out.astype("string")


def require_cols(df: pd.DataFrame, cols: list[str], name: str) -> None:
    """
    Stopper koden tidligt, hvis nødvendige kolonner mangler.
    """
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise KeyError(
            f"Mangler kolonner i {name}: {missing}. "
            f"Har: {df.columns.tolist()[:80]}"
        )


def pct(a: int | float, b: int | float) -> float:
    """
    Sikker procentfunktion.
    """
    return (100.0 * a / b) if b else float("nan")


def uniq_join(s: pd.Series, limit: int = 10) -> str:
    """
    Samler unikke værdier i en læsbar tekststreng til audit-eksport.
    """
    vals = sorted(map(str, pd.Series(s).dropna().unique()))
    txt = "; ".join(vals[:limit])
    if len(vals) > limit:
        txt += " ..."
    return txt


# ============================================================
# STEP 0 - LOCATE INPUTS
# ============================================================

print("STEP 0) Locate and unzip inputs ...")

if not BYG_ZIP.exists():
    raise FileNotFoundError(f"Mangler: {BYG_ZIP}")

if not GJ_ZIP.exists():
    raise FileNotFoundError(f"Mangler: {GJ_ZIP}")

mat_csvs = sorted(MAT_DIR.rglob("*.csv"), key=lambda p: p.stat().st_size, reverse=True)
if not mat_csvs:
    raise FileNotFoundError(f"Fandt ingen MAT CSV i {MAT_DIR}")

MAT_CSV = mat_csvs[0]

byg_csv = unzip_largest_csv(BYG_ZIP, DATA / "bygning_unzipped")
gj_csv  = unzip_largest_csv(GJ_ZIP, DATA / "grundjordstykke_unzipped")

prov = pd.DataFrame([
    {"label": "BBR_Bygning_zip", "path": str(BYG_ZIP), "sha256": sha256(BYG_ZIP)},
    {"label": "BBR_Bygning_csv", "path": str(byg_csv), "sha256": sha256(byg_csv)},
    {"label": "BBR_GrundJordstykke_zip", "path": str(GJ_ZIP), "sha256": sha256(GJ_ZIP)},
    {"label": "BBR_GrundJordstykke_csv", "path": str(gj_csv), "sha256": sha256(gj_csv)},
    {"label": "MAT_Jordstykke_csv", "path": str(MAT_CSV), "sha256": sha256(MAT_CSV)},
])

prov_out = EXPORT / "input_provenance.csv"
prov.to_csv(prov_out, index=False, encoding="utf-8")

print("✅ Provenance saved:", prov_out)
print("✅ Bygning CSV:", byg_csv)
print("✅ GrundJordstykke CSV:", gj_csv)
print("✅ MAT CSV:", MAT_CSV)


# ============================================================
# STEP 1 - LOAD DATA
# ============================================================

print("\nSTEP 1) Load CSVs ...")

byg = pd.read_csv(byg_csv, low_memory=False)
gj  = pd.read_csv(gj_csv, low_memory=False)

mat_cols_needed = [MAT_KEY_COL, "ejerlavLokalId", "matrikelnummer"]

mat = pd.read_csv(
    MAT_CSV,
    low_memory=False,
    usecols=lambda c: c in set(mat_cols_needed)
)

print("✅ Loaded byg:", len(byg), "rows |", len(byg.columns), "cols")
print("✅ Loaded gj :", len(gj),  "rows |", len(gj.columns), "cols")
print("✅ Loaded mat:", len(mat), "rows |", len(mat.columns), "cols")

require_cols(byg, ["grund", YEAR_COL, "id_lokalId"], "byg")
require_cols(gj, ["grund"], "gj")
require_cols(mat, mat_cols_needed, "mat")

# Kommune-filter, hvis kolonnen findes
if "kommunekode" in byg.columns:
    before = len(byg)
    byg = byg[byg["kommunekode"].astype(str).str.zfill(4) == KOMMUNE_KODE].copy()
    print(f"✅ Kommune-filter {KOMMUNE_KODE}: {len(byg)} rows tilbage af {before}")


# ============================================================
# STEP 2 - JOIN BYGNING -> GRUNDJORDSTYKKE
# ============================================================

print("\nSTEP 2) Join bygning -> grundjordstykke ...")

# Find jordstykke-kolonne i GJ
if "jordstykke" in gj.columns:
    gj_jord_col = "jordstykke"
else:
    cands = [c for c in gj.columns if "jordstykke" in c.lower()]
    if not cands:
        raise KeyError("Fandt ingen jordstykke-kolonne i gj.")
    gj_jord_col = sorted(cands, key=len)[0]

print("✅ Bruger GJ jordstykke-kolonne:", gj_jord_col)

right_gj = (
    gj.assign(
        _grund=norm_str(gj["grund"]),
        jordstykke_id=norm_join_key(gj[gj_jord_col])
    )[["_grund", "jordstykke_id"]]
    .dropna(subset=["_grund", "jordstykke_id"])
    .drop_duplicates()
)

bg = (
    byg.assign(
        _grund=norm_str(byg["grund"]),
        building_id=norm_str(byg["id_lokalId"]),
        opfoer=to_int(byg[YEAR_COL])
    )
    .merge(right_gj, on="_grund", how="left")
)

print(
    "✅ Bygning -> GJ jordstykke_id udfyldt:",
    int(bg["jordstykke_id"].notna().sum()),
    "af",
    len(bg),
    "rows"
)


# ============================================================
# STEP 3 - JOIN GRUNDJORDSTYKKE -> MATRIKEL
# ============================================================

print("\nSTEP 3) Join grundjordstykke -> MAT ...")

mat2 = mat.copy()

mat2["jordstykke_id"] = norm_join_key(mat2[MAT_KEY_COL])
mat2["ejerlavLokalId"] = norm_str(mat2["ejerlavLokalId"])
mat2["matrikelnummer_str"] = norm_str(mat2["matrikelnummer"])

mat2 = (
    mat2
    .dropna(subset=["jordstykke_id", "ejerlavLokalId", "matrikelnummer_str"])
    .drop_duplicates(["jordstykke_id", "ejerlavLokalId", "matrikelnummer_str"])
)

bgm = bg.merge(
    mat2[["jordstykke_id", "ejerlavLokalId", "matrikelnummer", "matrikelnummer_str"]],
    on="jordstykke_id",
    how="left"
)

match_rate_rows = float(bgm["matrikelnummer_str"].notna().mean())

match_rate_buildings = (
    bgm.groupby("building_id")["matrikelnummer_str"]
       .apply(lambda s: s.notna().any())
       .mean()
)

print("✅ GJ -> MAT match-rate rows:", round(match_rate_rows * 100, 2), "%")
print("✅ GJ -> MAT match-rate unique buildings:", round(match_rate_buildings * 100, 2), "%")


# ============================================================
# STEP 4 - CLEAN / VALID FILTER
# ============================================================

print("\nSTEP 4) Clean and filter valid rows ...")

required_valid = [
    "ejerlavLokalId",
    "matrikelnummer_str",
    "opfoer",
    "jordstykke_id",
    "building_id"
]

valid_raw = bgm.dropna(subset=required_valid).copy()

valid_raw = valid_raw[
    (valid_raw["opfoer"] >= MIN_PLAUSIBLE_YEAR)
    & (valid_raw["opfoer"] <= MAX_PLAUSIBLE_YEAR)
].copy()

before_dedup = len(valid_raw)

valid = valid_raw.drop_duplicates(
    subset=[
        "ejerlavLokalId",
        "matrikelnummer_str",
        "jordstykke_id",
        "building_id",
        "opfoer"
    ]
).copy()

removed_dupes = before_dedup - len(valid)

valid["matrikel_key"] = (
    valid["ejerlavLokalId"].astype("string")
    + "::"
    + valid["matrikelnummer_str"].astype("string")
)

print("✅ Valid rows før dedup:", before_dedup)
print("✅ Valid rows efter dedup:", len(valid))
print("✅ Fjernede eksakte dublet-rækker:", removed_dupes)
print("✅ Unikke bygninger i valid:", valid["building_id"].nunique())
print("✅ Unikke matrikler i valid:", valid["matrikel_key"].nunique())

if valid.empty:
    raise RuntimeError("valid er tom efter filtrering. Tjek join-nøgler, inputfiler og årskolonne.")


# ============================================================
# STEP 5 - AUDIT: BYGNINGER PÅ FLERE MATRIKLER
# ============================================================

print("\nSTEP 5) Audit buildings mapped to multiple matrikler ...")

audit_building_matrikler = (
    valid.groupby("building_id", dropna=False)
    .agg(
        antal_rækker=("building_id", "size"),
        antal_jordstykker=("jordstykke_id", "nunique"),
        antal_matrikler=("matrikel_key", "nunique"),
        min_year=("opfoer", "min"),
        max_year=("opfoer", "max"),
        matrikler=("matrikel_key", uniq_join),
    )
    .reset_index()
    .sort_values(["antal_matrikler", "antal_jordstykker", "antal_rækker"], ascending=False)
)

problem_buildings = audit_building_matrikler[
    audit_building_matrikler["antal_matrikler"] > 1
].copy()

problem_out = EXPORT / "audit_bygninger_paa_flere_matrikler.csv"
audit_all_out = EXPORT / "audit_alle_bygninger_matrikelkobling.csv"

problem_buildings.to_csv(problem_out, index=False, encoding="utf-8")
audit_building_matrikler.to_csv(audit_all_out, index=False, encoding="utf-8")

n_unique_buildings = valid["building_id"].nunique()
n_problem_buildings = problem_buildings["building_id"].nunique()

print(
    "⚠️ Bygninger koblet til flere matrikler:",
    n_problem_buildings,
    f"({pct(n_problem_buildings, n_unique_buildings):.2f}% af unikke bygninger)"
)

print("✅ Audit saved:", problem_out)
print("✅ Full audit saved:", audit_all_out)

if n_problem_buildings > 0:
    print("\nTop 10 bygninger koblet til flest matrikler:")
    display(problem_buildings.head(10))


# ============================================================
# STEP 6 - AGGREGATE PER MATRIKEL + CONSISTENT CUT-OFF
# ============================================================

print("\nSTEP 6) Aggregate per matrikel and classify ...")

grp = (
    valid.groupby(["ejerlavLokalId", "matrikelnummer_str"], dropna=False)
    .agg(
        min_year=("opfoer", "min"),
        max_year=("opfoer", "max"),
        unikke_bygninger=("building_id", "nunique"),
        unikke_jordstykker=("jordstykke_id", "nunique"),
    )
    .reset_index()
)

grp["matrikel_key"] = (
    grp["ejerlavLokalId"].astype("string")
    + "::"
    + grp["matrikelnummer_str"].astype("string")
)

grp["klasse"] = "ukendt"

grp.loc[
    grp["max_year"] <= OLD_MAX_YEAR,
    "klasse"
] = "kun_gammel"

grp.loc[
    grp["min_year"] >= NEW_MIN_YEAR,
    "klasse"
] = "kun_ny"

grp.loc[
    (grp["min_year"] <= OLD_MAX_YEAR)
    & (grp["max_year"] >= NEW_MIN_YEAR),
    "klasse"
] = "begge"

if (grp["klasse"] == "ukendt").any():
    bad = grp[grp["klasse"] == "ukendt"].head(20)
    raise RuntimeError(
        "Fandt matrikler der ikke passer i kun_gammel/kun_ny/begge. "
        "Eksempler:\n" + bad.to_string(index=False)
    )

klasse_counts = (
    grp["klasse"]
    .value_counts()
    .rename_axis("klasse")
    .reset_index(name="antal_matrikler")
)

both = (
    grp[grp["klasse"] == "begge"]
    .copy()
    .sort_values(["unikke_bygninger", "max_year"], ascending=[False, False])
)

print("✅ Antal matrikler i alt:", len(grp))
print("✅ Antal 'begge'-matrikler:", len(both))
print("\nMatrikler fordelt på klasse:")
display(klasse_counts)


# ============================================================
# STEP 7 - PUSH CLASS BACK TO BUILDING LEVEL
# ============================================================

print("\nSTEP 7) Push matrikel class back to building level ...")

v = valid.merge(
    grp[["ejerlavLokalId", "matrikelnummer_str", "klasse"]],
    on=["ejerlavLokalId", "matrikelnummer_str"],
    how="left",
    validate="m:1"
)

v["is_new_building"] = v["opfoer"] >= NEW_MIN_YEAR
v["is_old_building"] = v["opfoer"] <= OLD_MAX_YEAR

v["new_building_id"] = v["building_id"].where(v["is_new_building"])
v["old_building_id"] = v["building_id"].where(v["is_old_building"])

if v["klasse"].isna().any():
    raise RuntimeError("Nogle building-level rows fik ikke klasse efter merge.")


# ============================================================
# STEP 8 - COUNTS AND SMITTE-RISK SUMMARY
# ============================================================

print("\nSTEP 8) Count unique buildings and new-build distribution ...")

unikke_total = v["building_id"].nunique()
unikke_ny = v.loc[v["is_new_building"], "building_id"].nunique()
unikke_gammel = v.loc[v["is_old_building"], "building_id"].nunique()

nybyg_kun_ny = v.loc[
    (v["klasse"] == "kun_ny") & (v["is_new_building"]),
    "building_id"
].nunique()

nybyg_begge = v.loc[
    (v["klasse"] == "begge") & (v["is_new_building"]),
    "building_id"
].nunique()

nybyg_kun_gammel = v.loc[
    (v["klasse"] == "kun_gammel") & (v["is_new_building"]),
    "building_id"
].nunique()

buildings_by_class = (
    v.groupby("klasse", dropna=False)
    .agg(
        unikke_bygninger_total=("building_id", "nunique"),
        unikke_nybyg_bygninger=("new_building_id", "nunique"),
        unikke_gamle_bygninger=("old_building_id", "nunique"),
        antal_rækker=("building_id", "size"),
        unikke_matrikler=("matrikel_key", "nunique"),
        unikke_jordstykker=("jordstykke_id", "nunique"),
    )
    .reset_index()
    .sort_values("unikke_nybyg_bygninger", ascending=False)
)

summary = pd.DataFrame([
    {
        "metric": "unikke_bygninger_total",
        "value": unikke_total,
        "pct_af_nybyg": None,
    },
    {
        "metric": "unikke_gamle_bygninger_opfoer <= 1991",
        "value": unikke_gammel,
        "pct_af_nybyg": None,
    },
    {
        "metric": "unikke_nybyg_bygninger_opfoer >= 1992",
        "value": unikke_ny,
        "pct_af_nybyg": 100.0,
    },
    {
        "metric": "nybyg_paa_kun_ny_matrikler",
        "value": nybyg_kun_ny,
        "pct_af_nybyg": pct(nybyg_kun_ny, unikke_ny),
    },
    {
        "metric": "nybyg_paa_begge_matrikler_smitterisiko",
        "value": nybyg_begge,
        "pct_af_nybyg": pct(nybyg_begge, unikke_ny),
    },
    {
        "metric": "nybyg_paa_kun_gammel_matrikler_burde_vaere_0",
        "value": nybyg_kun_gammel,
        "pct_af_nybyg": pct(nybyg_kun_gammel, unikke_ny),
    },
    {
        "metric": "bygninger_koblet_til_flere_matrikler_audit",
        "value": n_problem_buildings,
        "pct_af_nybyg": None,
    },
])

print("\n=== CONSISTENT CUT-OFF ===")
print(f"Old <= {OLD_MAX_YEAR}")
print(f"New >= {NEW_MIN_YEAR}")
print()
print("Unikke bygninger i valid total:", unikke_total)
print("Unikke gamle bygninger opført <= 1991:", unikke_gammel)
print("Unikke nybyg-bygninger opført >= 1992:", unikke_ny)
print()
print(
    "Nybyg på kun_ny matrikler:",
    nybyg_kun_ny,
    f"({pct(nybyg_kun_ny, unikke_ny):.2f}%)"
)
print(
    "Nybyg på begge matrikler, proxy for smitterisiko:",
    nybyg_begge,
    f"({pct(nybyg_begge, unikke_ny):.2f}%)"
)
print(
    "Nybyg på kun_gammel matrikler, bør være 0:",
    nybyg_kun_gammel,
    f"({pct(nybyg_kun_gammel, unikke_ny):.2f}%)"
)

print("\nBygninger pr. klasse:")
display(buildings_by_class)

print("\nSummary:")
display(summary)


# ============================================================
# STEP 9 - LEAK CHECK
# ============================================================

print("\nSTEP 9) Leak check ...")

leaks = v[
    (v["klasse"] == "kun_gammel")
    & (v["is_new_building"])
][
    [
        "ejerlavLokalId",
        "matrikelnummer_str",
        "jordstykke_id",
        "building_id",
        "opfoer",
        "klasse",
    ]
].drop_duplicates()

leaks_out = EXPORT / "audit_leaks_nybyg_paa_kun_gammel.csv"
leaks.to_csv(leaks_out, index=False, encoding="utf-8")

if len(leaks) > 0:
    print("⚠️ Fandt nybyg på kun_gammel matrikler. Tjek data.")
    display(leaks.head(20))
else:
    print("✅ Ingen nybyg på kun_gammel matrikler.")

print("✅ Leak audit saved:", leaks_out)


# ============================================================
# STEP 10 - EXPORT TABLES
# ============================================================

print("\nSTEP 10) Export tables ...")

out_both = EXPORT / f"matrikler_begge_cutoff_{CUT_YEAR}_years_{MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR}.csv"
out_grp  = EXPORT / f"matrikler_alle_klasser_cutoff_{CUT_YEAR}_years_{MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR}.csv"
out_cls  = EXPORT / f"klasse_fordeling_cutoff_{CUT_YEAR}_years_{MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR}.csv"
out_buildings_by_class = EXPORT / f"bygninger_pr_klasse_cutoff_{CUT_YEAR}.csv"
out_summary = EXPORT / f"summary_cutoff_{CUT_YEAR}.csv"

both.to_csv(out_both, index=False, encoding="utf-8")
grp.to_csv(out_grp, index=False, encoding="utf-8")
klasse_counts.to_csv(out_cls, index=False, encoding="utf-8")
buildings_by_class.to_csv(out_buildings_by_class, index=False, encoding="utf-8")
summary.to_csv(out_summary, index=False, encoding="utf-8")

if EXPORT_VALID_BUILDING_LEVEL:
    out_valid = EXPORT / f"building_level_valid_classified_cutoff_{CUT_YEAR}.csv"
    v.to_csv(out_valid, index=False, encoding="utf-8")
    print("✅ saved:", out_valid)

print("✅ saved:", out_both)
print("✅ saved:", out_grp)
print("✅ saved:", out_cls)
print("✅ saved:", out_buildings_by_class)
print("✅ saved:", out_summary)


# ============================================================
# STEP 11 - PLOTS
# ============================================================

print("\nSTEP 11) Create plots ...")

fig1 = EXPORT / "hist_min_year.png"
fig2 = EXPORT / "klasse_fordeling.png"
fig3 = EXPORT / "nybyg_bygninger_pr_klasse.png"

plt.figure()
grp["min_year"].dropna().astype(int).hist(bins=50)
plt.title(f"Fordeling af tidligste opførelsesår pr. matrikel ({MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR})")
plt.xlabel("Min opførelsesår")
plt.ylabel("Antal matrikler")
plt.savefig(fig1, dpi=200, bbox_inches="tight")
plt.close()

plt.figure()
klasse_counts.set_index("klasse")["antal_matrikler"].plot(kind="bar")
plt.title(f"Matrikler fordelt på klasse, cutoff={CUT_YEAR}")
plt.xlabel("Klasse")
plt.ylabel("Antal matrikler")
plt.savefig(fig2, dpi=200, bbox_inches="tight")
plt.close()

plt.figure()
buildings_by_class.set_index("klasse")["unikke_nybyg_bygninger"].plot(kind="bar")
plt.title(f"Nybyg-bygninger fordelt på matrikelklasse, cutoff={CUT_YEAR}")
plt.xlabel("Klasse")
plt.ylabel("Unikke nybyg-bygninger")
plt.savefig(fig3, dpi=200, bbox_inches="tight")
plt.close()

print("✅ saved figs:", fig1, fig2, fig3)


# ============================================================
# STEP 12 - FINAL OUTPUT IN NOTEBOOK
# ============================================================

print("\n=== DONE ===")
print("Vigtigste output-objekter i notebook:")
print("- valid: renset building/matrikel-level datasæt før klasse-merge")
print("- v: building/matrikel-level datasæt med klasse og ny/gammel flags")
print("- grp: matrikel-level datasæt med min_year, max_year og klasse")
print("- both: matrikler med både gamle og nye bygninger")
print("- buildings_by_class: bygningstællinger pr. klasse")
print("- problem_buildings: bygninger koblet til flere matrikler")
print("- summary: samlet hovedresultat")

both.head(10)

STEP 0) Locate and unzip inputs ...
✅ Provenance saved: exports/input_provenance.csv
✅ Bygning CSV: data/bygning_unzipped/BBR_V3_Bygning_0101_TotalDownload_csv_Current_535.csv
✅ GrundJordstykke CSV: data/grundjordstykke_unzipped/BBR_V3_GrundJordstykke_0101_TotalDownload_csv_Current_535.csv
✅ MAT CSV: data/mat_jordstykke/MAT_V3_Jordstykke_TotalDownload_csv_Temporal_574.csv

STEP 1) Load CSVs ...


In [ ]:
# ============================================================
# FORTSÆTTELSE EFTER enh ER LOADED
# Filtrerer bygninger til udlejningsejendomme til beboelse
# ============================================================

# Snæver definition:
# 140 = etagebolig/flerfamiliehus/to-familiehus
BUILDING_RESIDENTIAL_CODES = {140}
UNIT_RESIDENTIAL_CODES = {140}

# 1 = Udlejet
RENTED_CODES = {1}

# Brug 1 for bred definition, 2 for mere klassisk udlejningsejendom
MIN_RENTED_RESIDENTIAL_UNITS = 2

# Kræv at selve bygningen også er BBR-kode 140
REQUIRE_BUILDING_CODE_140 = True


# ------------------------------------------------------------
# Kommune-filter på Enhed
# ------------------------------------------------------------

if "kommunekode" in enh.columns:
    before_enh_kommune = len(enh)

    enh = enh[
        enh["kommunekode"].astype(str).str.zfill(4) == KOMMUNE_KODE
    ].copy()

    print(
        f"✅ Enhed kommune-filter {KOMMUNE_KODE}:",
        len(enh),
        "rows tilbage af",
        before_enh_kommune
    )


# ------------------------------------------------------------
# Normalisér nøgler og koder
# ------------------------------------------------------------

byg["building_id"] = norm_str(byg["id_lokalId"])

byg["_byg_anvendelse"] = pd.to_numeric(
    byg["byg021BygningensAnvendelse"],
    errors="coerce"
).astype("Int64")

enh["building_id"] = norm_str(enh["bygning"])

enh["_enh_anvendelse"] = pd.to_numeric(
    enh["enh020EnhedensAnvendelse"],
    errors="coerce"
).astype("Int64")

enh["_udlejning"] = pd.to_numeric(
    enh["enh045Udlejningsforhold"],
    errors="coerce"
).astype("Int64")

# Arealer gøres numeriske, så aggregation ikke fejler
area_cols = [
    "enh026EnhedensSamledeAreal",
    "enh027ArealTilBeboelse",
    "enh028ArealTilErhverv",
]

for c in area_cols:
    if c in enh.columns:
        enh[c] = pd.to_numeric(enh[c], errors="coerce").fillna(0)


# ------------------------------------------------------------
# Audit: se først kodefordelingerne
# ------------------------------------------------------------

print("\nFordeling i Bygning - byg021BygningensAnvendelse:")
display(
    byg["_byg_anvendelse"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("byg021")
    .reset_index(name="antal_bygninger")
)

print("\nFordeling i Enhed - enh020EnhedensAnvendelse:")
display(
    enh["_enh_anvendelse"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("enh020")
    .reset_index(name="antal_enheder")
)

print("\nFordeling i Enhed - enh045Udlejningsforhold:")
display(
    enh["_udlejning"]
    .value_counts(dropna=False)
    .rename_axis("enh045")
    .reset_index(name="antal_enheder")
)


# ------------------------------------------------------------
# Flags
# ------------------------------------------------------------

byg["is_relevant_residential_building"] = (
    byg["_byg_anvendelse"].isin(BUILDING_RESIDENTIAL_CODES)
)

enh["is_relevant_residential_unit"] = (
    enh["_enh_anvendelse"].isin(UNIT_RESIDENTIAL_CODES)
)

enh["is_rented_unit"] = (
    enh["_udlejning"].isin(RENTED_CODES)
)

enh["is_rented_residential_unit"] = (
    enh["is_relevant_residential_unit"]
    & enh["is_rented_unit"]
)


# ------------------------------------------------------------
# Aggregér Enhed op til Bygning
# ------------------------------------------------------------

enh_unit_summary = (
    enh.groupby("building_id", dropna=False)
    .agg(
        antal_enheder=("id_lokalId", "nunique"),
        antal_relevante_boligenheder=("is_relevant_residential_unit", "sum"),
        antal_udlejede_enheder=("is_rented_unit", "sum"),
        antal_udlejede_relevante_boligenheder=("is_rented_residential_unit", "sum"),
        samlet_enhedsareal=("enh026EnhedensSamledeAreal", "sum"),
        samlet_boligareal_enheder=("enh027ArealTilBeboelse", "sum"),
        samlet_erhvervsareal_enheder=("enh028ArealTilErhverv", "sum"),
    )
    .reset_index()
)

enh_unit_summary["andel_udlejede_relevante_boligenheder"] = (
    enh_unit_summary["antal_udlejede_relevante_boligenheder"]
    / enh_unit_summary["antal_relevante_boligenheder"].replace(0, pd.NA)
)


# ------------------------------------------------------------
# Merge Enhed-summary på Bygning
# ------------------------------------------------------------

before_merge = len(byg)

byg = byg.merge(
    enh_unit_summary,
    on="building_id",
    how="left",
    validate="1:1"
)

count_cols = [
    "antal_enheder",
    "antal_relevante_boligenheder",
    "antal_udlejede_enheder",
    "antal_udlejede_relevante_boligenheder",
    "samlet_enhedsareal",
    "samlet_boligareal_enheder",
    "samlet_erhvervsareal_enheder",
]

for c in count_cols:
    byg[c] = byg[c].fillna(0)

print("\n✅ Bygninger før Enhed-merge:", before_merge)
print("✅ Bygninger efter Enhed-merge:", len(byg))
print("✅ Bygninger med mindst én Enhed:", int((byg["antal_enheder"] > 0).sum()))


# ------------------------------------------------------------
# Filtrér til udlejningsejendomme til beboelse
# ------------------------------------------------------------

if REQUIRE_BUILDING_CODE_140:
    rental_filter = (
        byg["is_relevant_residential_building"]
        & (
            byg["antal_udlejede_relevante_boligenheder"]
            >= MIN_RENTED_RESIDENTIAL_UNITS
        )
    )
else:
    rental_filter = (
        byg["antal_udlejede_relevante_boligenheder"]
        >= MIN_RENTED_RESIDENTIAL_UNITS
    )

before_rental_filter = len(byg)

byg = byg[rental_filter].copy()

print("\n✅ FILTER: Udlejningsejendomme til beboelse")
print("Krav om byg021 == 140:", REQUIRE_BUILDING_CODE_140)
print("Bygningskoder:", BUILDING_RESIDENTIAL_CODES)
print("Enhedskoder:", UNIT_RESIDENTIAL_CODES)
print("Udlejningskoder:", RENTED_CODES)
print("Minimum udlejede relevante boligenheder:", MIN_RENTED_RESIDENTIAL_UNITS)
print("Rows før filter:", before_rental_filter)
print("Rows efter filter:", len(byg))
print("Fjernet:", before_rental_filter - len(byg))

print("\nNøgletal for den filtrerede population:")
display(
    byg[
        [
            "building_id",
            "_byg_anvendelse",
            "antal_enheder",
            "antal_relevante_boligenheder",
            "antal_udlejede_enheder",
            "antal_udlejede_relevante_boligenheder",
            "andel_udlejede_relevante_boligenheder",
            "samlet_boligareal_enheder",
            "samlet_erhvervsareal_enheder",
        ]
    ].describe(include="all")
)

In [1]:
# ============================================================
# SAMLET PIPELINE:
# UDLEJNINGSEJENDOMME TIL BEBOELSE
# + BBR ENHED FILTER
# + MATRIKELKLASSIFIKATION
# + CONSISTENT CUT-OFF 1991/1992
# + AUDIT OG EXPORT
# ============================================================

from __future__ import annotations

from pathlib import Path
import zipfile
import hashlib
import pandas as pd
import matplotlib.pyplot as plt


# ============================================================
# CONFIG
# ============================================================

KOMMUNE_KODE = "0101"

# Cut-off:
# Gammel: opført senest 1991
# Ny: opført fra og med 1992
OLD_MAX_YEAR = 1991
NEW_MIN_YEAR = 1992
CUT_YEAR = NEW_MIN_YEAR

MIN_PLAUSIBLE_YEAR = 1500
MAX_PLAUSIBLE_YEAR = 2026

YEAR_COL = "byg026Opførelsesår"

DATA = Path("data")
EXPORT = Path("exports")
DATA.mkdir(exist_ok=True)
EXPORT.mkdir(exist_ok=True)

# MAT-nøgle, som du tidligere har låst
MAT_KEY_COL = "id_lokalId"

# ------------------------------------------------------------
# Definition af udlejningsejendom til beboelse
# ------------------------------------------------------------

# Bygning:
# 140 = Etagebolig-bygning, flerfamiliehus eller to-familiehus
BUILDING_RESIDENTIAL_CODES = {140}

# Enhed:
# 140 = Bolig i etageejendom, flerfamiliehus eller to-familiehus
UNIT_RESIDENTIAL_CODES = {140}

# Udlejningsforhold:
# 1 = Udlejet
RENTED_CODES = {1}

# Brug 1 for bred definition.
# Brug 2 for mere klassisk udlejningsejendom og for at undgå enkelte udlejede boliger.
MIN_RENTED_RESIDENTIAL_UNITS = 2

# Hvis True kræves både:
# - byg021 == 140
# - mindst X udlejede relevante boligenheder
#
# Hvis False bruges kun Enhed-filteret.
REQUIRE_BUILDING_CODE_140 = True

# Hvis True eksporteres hele building-level datasættet.
# Det kan være stort.
EXPORT_VALID_BUILDING_LEVEL = False


# ============================================================
# HELPERS
# ============================================================

def unzip_largest_csv(zip_path: Path, out_dir: Path) -> Path:
    """
    Udpakker en zip-fil og returnerer den største CSV-fil.
    """
    out_dir.mkdir(parents=True, exist_ok=True)

    with zipfile.ZipFile(zip_path, "r") as z:
        z.extractall(out_dir)

    csvs = list(out_dir.rglob("*.csv"))
    if not csvs:
        raise RuntimeError(f"Fandt ingen CSV i {zip_path}")

    csvs.sort(key=lambda p: p.stat().st_size, reverse=True)
    return csvs[0]


def sha256(path: Path, chunk: int = 1024 * 1024) -> str:
    """
    Laver SHA256-hash til provenance.
    """
    h = hashlib.sha256()

    with open(path, "rb") as f:
        while True:
            b = f.read(chunk)
            if not b:
                break
            h.update(b)

    return h.hexdigest()


def norm_str(s: pd.Series) -> pd.Series:
    """
    Normaliserer tekstkolonne.
    """
    return s.astype("string").str.strip()


def to_int(s: pd.Series) -> pd.Series:
    """
    Konverterer til nullable integer.
    """
    return pd.to_numeric(s, errors="coerce").astype("Int64")


def norm_join_key(s: pd.Series) -> pd.Series:
    """
    Robust normalisering af join-nøgler.

    Hvis værdien er numerisk, konverteres den til integer-string.
    Hvis den ikke er numerisk, beholdes trimmed string.
    """
    raw = norm_str(s)

    empty_mask = (
        raw.isna()
        | raw.eq("")
        | raw.str.lower().isin(["nan", "none", "<na>"])
    )

    raw_clean = raw.str.replace(r"\.0$", "", regex=True)
    numeric = pd.to_numeric(raw_clean, errors="coerce")

    out = raw_clean.copy()
    numeric_mask = numeric.notna()

    out.loc[numeric_mask] = numeric.loc[numeric_mask].astype("Int64").astype("string")
    out.loc[empty_mask] = pd.NA

    return out.astype("string")


def require_cols(df: pd.DataFrame, cols: list[str], name: str) -> None:
    """
    Stopper koden, hvis nødvendige kolonner mangler.
    """
    missing = [c for c in cols if c not in df.columns]

    if missing:
        raise KeyError(
            f"Mangler kolonner i {name}: {missing}. "
            f"Har: {df.columns.tolist()[:100]}"
        )


def pct(a: int | float, b: int | float) -> float:
    """
    Sikker procentfunktion.
    """
    return (100.0 * a / b) if b else float("nan")


def uniq_join(s: pd.Series, limit: int = 10) -> str:
    """
    Samler unikke værdier til audit.
    """
    vals = sorted(map(str, pd.Series(s).dropna().unique()))
    txt = "; ".join(vals[:limit])

    if len(vals) > limit:
        txt += " ..."

    return txt


def find_file(patterns: list[str], root: Path = DATA, prefer_largest: bool = True) -> Path:
    """
    Finder en fil ud fra flere patterns.
    """
    candidates = []

    for pattern in patterns:
        candidates.extend(root.rglob(pattern))

    candidates = [p for p in candidates if p.is_file()]

    # Undgå checkpoint-filer
    candidates = [p for p in candidates if ".ipynb_checkpoints" not in str(p)]

    if not candidates:
        print("Filer i data-mappen:")
        for p in sorted(root.rglob("*")):
            if p.is_file():
                print("-", p)

        raise FileNotFoundError(
            f"Fandt ingen fil med patterns: {patterns}"
        )

    if prefer_largest:
        candidates.sort(key=lambda p: p.stat().st_size, reverse=True)
    else:
        candidates.sort()

    return candidates[0]


def csv_from_csv_or_zip(path: Path, unzip_dir: Path) -> Path:
    """
    Returnerer CSV-sti.
    Hvis input er zip, udpakkes største CSV.
    Hvis input er csv, returneres input.
    """
    if path.suffix.lower() == ".zip":
        return unzip_largest_csv(path, unzip_dir)

    if path.suffix.lower() == ".csv":
        return path

    raise ValueError(f"Filtypen understøttes ikke: {path}")


# ============================================================
# STEP 0 - FIND INPUTFILER
# ============================================================

print("STEP 0) Find inputfiler ...")

byg_path = find_file([
    "*BBR_V3_Bygning_0101*Current*.csv",
    "*BBR_V3_Bygning_0101*Current*.zip",
    "*Bygning_0101*.csv",
    "*Bygning_0101*.zip",
])

gj_path = find_file([
    "*BBR_V3_GrundJordstykke_0101*Current*.csv",
    "*BBR_V3_GrundJordstykke_0101*Current*.zip",
    "*GrundJordstykke_0101*.csv",
    "*GrundJordstykke_0101*.zip",
])

enh_path = find_file([
    "*BBR_V3_Enhed_0101*Current*.csv",
    "*BBR_V3_Enhed_0101*Current*.zip",
    "*Enhed_0101*.csv",
    "*Enhed_0101*.zip",
])

mat_path = find_file([
    "mat_jordstykke/*MAT_V3_Jordstykke*.csv",
    "mat_jordstykke/*MAT_V3_Jordstykke*.zip",
    "*MAT_V3_Jordstykke*.csv",
    "*MAT_V3_Jordstykke*.zip",
])

byg_csv = csv_from_csv_or_zip(byg_path, DATA / "bygning_unzipped")
gj_csv = csv_from_csv_or_zip(gj_path, DATA / "grundjordstykke_unzipped")
enh_csv = csv_from_csv_or_zip(enh_path, DATA / "enhed_unzipped")
mat_csv = csv_from_csv_or_zip(mat_path, DATA / "mat_jordstykke_unzipped")

print("✅ Bygning:", byg_csv)
print("✅ GrundJordstykke:", gj_csv)
print("✅ Enhed:", enh_csv)
print("✅ MAT Jordstykke:", mat_csv)

prov = pd.DataFrame([
    {"label": "BBR_Bygning", "path": str(byg_csv), "sha256": sha256(byg_csv)},
    {"label": "BBR_GrundJordstykke", "path": str(gj_csv), "sha256": sha256(gj_csv)},
    {"label": "BBR_Enhed", "path": str(enh_csv), "sha256": sha256(enh_csv)},
    {"label": "MAT_Jordstykke", "path": str(mat_csv), "sha256": sha256(mat_csv)},
])

prov_out = EXPORT / "input_provenance.csv"
prov.to_csv(prov_out, index=False, encoding="utf-8")

print("✅ Provenance saved:", prov_out)


# ============================================================
# STEP 1 - LOAD DATA
# ============================================================

print("\nSTEP 1) Load CSV-filer ...")

byg_cols_needed = [
    "id_lokalId",
    "kommunekode",
    "grund",
    "byg021BygningensAnvendelse",
    YEAR_COL,
]

enh_cols_needed = [
    "id_lokalId",
    "kommunekode",
    "status",
    "bygning",
    "enh020EnhedensAnvendelse",
    "enh023Boligtype",
    "enh045Udlejningsforhold",
    "enh026EnhedensSamledeAreal",
    "enh027ArealTilBeboelse",
    "enh028ArealTilErhverv",
]

mat_cols_needed = [
    MAT_KEY_COL,
    "ejerlavLokalId",
    "matrikelnummer",
]

byg = pd.read_csv(
    byg_csv,
    low_memory=False,
    encoding="utf-8",
    usecols=lambda c: c in set(byg_cols_needed)
)

gj = pd.read_csv(
    gj_csv,
    low_memory=False,
    encoding="utf-8"
)

enh = pd.read_csv(
    enh_csv,
    low_memory=False,
    encoding="utf-8",
    usecols=lambda c: c in set(enh_cols_needed)
)

mat = pd.read_csv(
    mat_csv,
    low_memory=False,
    encoding="utf-8",
    usecols=lambda c: c in set(mat_cols_needed)
)

print("✅ Loaded byg:", len(byg), "rows |", len(byg.columns), "cols")
print("✅ Loaded gj :", len(gj), "rows |", len(gj.columns), "cols")
print("✅ Loaded enh:", len(enh), "rows |", len(enh.columns), "cols")
print("✅ Loaded mat:", len(mat), "rows |", len(mat.columns), "cols")

require_cols(byg, ["id_lokalId", "grund", "byg021BygningensAnvendelse", YEAR_COL], "byg")
require_cols(gj, ["grund"], "gj")
require_cols(enh, ["id_lokalId", "bygning", "enh020EnhedensAnvendelse", "enh045Udlejningsforhold"], "enh")
require_cols(mat, [MAT_KEY_COL, "ejerlavLokalId", "matrikelnummer"], "mat")


# ============================================================
# STEP 1A - KOMMUNEFILTER
# ============================================================

print("\nSTEP 1A) Kommune-filter ...")

if "kommunekode" in byg.columns:
    before = len(byg)
    byg = byg[byg["kommunekode"].astype(str).str.zfill(4) == KOMMUNE_KODE].copy()
    print(f"✅ Byg kommune-filter {KOMMUNE_KODE}: {len(byg)} rows tilbage af {before}")

if "kommunekode" in enh.columns:
    before = len(enh)
    enh = enh[enh["kommunekode"].astype(str).str.zfill(4) == KOMMUNE_KODE].copy()
    print(f"✅ Enhed kommune-filter {KOMMUNE_KODE}: {len(enh)} rows tilbage af {before}")


# ============================================================
# STEP 1B - FILTER TIL UDLEJNINGSEJENDOMME TIL BEBOELSE
# ============================================================

print("\nSTEP 1B) Filtrér til udlejningsejendomme til beboelse ...")

# Normalisér bygning
byg["building_id"] = norm_str(byg["id_lokalId"])

byg["_byg_anvendelse"] = pd.to_numeric(
    byg["byg021BygningensAnvendelse"],
    errors="coerce"
).astype("Int64")

byg["opfoer"] = pd.to_numeric(
    byg[YEAR_COL],
    errors="coerce"
).astype("Int64")

# Normalisér enhed
enh["building_id"] = norm_str(enh["bygning"])

enh["_enh_anvendelse"] = pd.to_numeric(
    enh["enh020EnhedensAnvendelse"],
    errors="coerce"
).astype("Int64")

enh["_udlejning_num"] = pd.to_numeric(
    enh["enh045Udlejningsforhold"],
    errors="coerce"
).astype("Int64")

enh["_udlejning_txt"] = norm_str(enh["enh045Udlejningsforhold"]).str.lower()

# Arealer
area_cols = [
    "enh026EnhedensSamledeAreal",
    "enh027ArealTilBeboelse",
    "enh028ArealTilErhverv",
]

for c in area_cols:
    if c in enh.columns:
        enh[c] = pd.to_numeric(enh[c], errors="coerce").fillna(0)

# Flags på Bygning
byg["is_relevant_residential_building"] = (
    byg["_byg_anvendelse"].isin(BUILDING_RESIDENTIAL_CODES)
)

# Flags på Enhed
enh["is_relevant_residential_unit"] = (
    enh["_enh_anvendelse"].isin(UNIT_RESIDENTIAL_CODES)
)

# Udlejning.
# Primært numerisk kode 1.
# Tekstfallback bruges kun ved præcis tekst eller tekst der starter med 1.
enh["is_rented_unit"] = (
    enh["_udlejning_num"].isin(RENTED_CODES)
    | enh["_udlejning_txt"].eq("udlejet")
    | enh["_udlejning_txt"].str.match(r"^1\b", na=False)
)

enh["is_rented_residential_unit"] = (
    enh["is_relevant_residential_unit"]
    & enh["is_rented_unit"]
)

print("\nFordeling i Bygning - byg021BygningensAnvendelse:")
display(
    byg["_byg_anvendelse"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("byg021")
    .reset_index(name="antal_bygninger")
)

print("\nFordeling i Enhed - enh020EnhedensAnvendelse:")
display(
    enh["_enh_anvendelse"]
    .value_counts(dropna=False)
    .head(20)
    .rename_axis("enh020")
    .reset_index(name="antal_enheder")
)

print("\nFordeling i Enhed - enh045Udlejningsforhold:")
display(
    enh["_udlejning_num"]
    .value_counts(dropna=False)
    .rename_axis("enh045")
    .reset_index(name="antal_enheder")
)

# Aggregér Enhed op til Bygning
enh_unit_summary = (
    enh.groupby("building_id", dropna=False)
    .agg(
        antal_enheder=("id_lokalId", "nunique"),
        antal_relevante_boligenheder=("is_relevant_residential_unit", "sum"),
        antal_udlejede_enheder=("is_rented_unit", "sum"),
        antal_udlejede_relevante_boligenheder=("is_rented_residential_unit", "sum"),
        samlet_enhedsareal=("enh026EnhedensSamledeAreal", "sum"),
        samlet_boligareal_enheder=("enh027ArealTilBeboelse", "sum"),
        samlet_erhvervsareal_enheder=("enh028ArealTilErhverv", "sum"),
    )
    .reset_index()
)

enh_unit_summary["andel_udlejede_relevante_boligenheder"] = (
    enh_unit_summary["antal_udlejede_relevante_boligenheder"]
    / enh_unit_summary["antal_relevante_boligenheder"].replace(0, pd.NA)
)

# Merge Enhed-summary på Bygning
before_merge = len(byg)

byg = byg.merge(
    enh_unit_summary,
    on="building_id",
    how="left",
    validate="1:1"
)

enhed_count_cols = [
    "antal_enheder",
    "antal_relevante_boligenheder",
    "antal_udlejede_enheder",
    "antal_udlejede_relevante_boligenheder",
    "samlet_enhedsareal",
    "samlet_boligareal_enheder",
    "samlet_erhvervsareal_enheder",
]

for c in enhed_count_cols:
    byg[c] = byg[c].fillna(0)

print("\n✅ Bygninger før Enhed-merge:", before_merge)
print("✅ Bygninger efter Enhed-merge:", len(byg))
print("✅ Bygninger med mindst én Enhed:", int((byg["antal_enheder"] > 0).sum()))

# Definér analysepopulation
if REQUIRE_BUILDING_CODE_140:
    rental_filter = (
        byg["is_relevant_residential_building"]
        & (
            byg["antal_udlejede_relevante_boligenheder"]
            >= MIN_RENTED_RESIDENTIAL_UNITS
        )
    )
else:
    rental_filter = (
        byg["antal_udlejede_relevante_boligenheder"]
        >= MIN_RENTED_RESIDENTIAL_UNITS
    )

before_rental_filter = len(byg)

byg = byg[rental_filter].copy()

print("\n✅ FILTER: Udlejningsejendomme til beboelse")
print("Krav om byg021 == 140:", REQUIRE_BUILDING_CODE_140)
print("Bygningskoder:", BUILDING_RESIDENTIAL_CODES)
print("Enhedskoder:", UNIT_RESIDENTIAL_CODES)
print("Udlejningskoder:", RENTED_CODES)
print("Minimum udlejede relevante boligenheder:", MIN_RENTED_RESIDENTIAL_UNITS)
print("Rows før filter:", before_rental_filter)
print("Rows efter filter:", len(byg))
print("Fjernet:", before_rental_filter - len(byg))

if byg.empty:
    raise RuntimeError(
        "Filteret gav 0 bygninger. "
        "Prøv MIN_RENTED_RESIDENTIAL_UNITS = 1 eller REQUIRE_BUILDING_CODE_140 = False."
    )

print("\nNøgletal for filtreret population:")
display(
    byg[
        [
            "building_id",
            "_byg_anvendelse",
            "opfoer",
            "antal_enheder",
            "antal_relevante_boligenheder",
            "antal_udlejede_enheder",
            "antal_udlejede_relevante_boligenheder",
            "andel_udlejede_relevante_boligenheder",
            "samlet_boligareal_enheder",
            "samlet_erhvervsareal_enheder",
        ]
    ].describe(include="all")
)


# ============================================================
# STEP 2 - JOIN BYGNING -> GRUNDJORDSTYKKE
# ============================================================

print("\nSTEP 2) Join bygning -> grundjordstykke ...")

if "jordstykke" in gj.columns:
    gj_jord_col = "jordstykke"
else:
    cands = [c for c in gj.columns if "jordstykke" in c.lower()]

    if not cands:
        raise KeyError("Fandt ingen jordstykke-kolonne i gj.")

    gj_jord_col = sorted(cands, key=len)[0]

print("✅ Bruger GJ jordstykke-kolonne:", gj_jord_col)

right_gj = (
    gj.assign(
        _grund=norm_str(gj["grund"]),
        jordstykke_id=norm_join_key(gj[gj_jord_col])
    )[["_grund", "jordstykke_id"]]
    .dropna(subset=["_grund", "jordstykke_id"])
    .drop_duplicates()
)

bg = (
    byg.assign(
        _grund=norm_str(byg["grund"])
    )
    .merge(
        right_gj,
        on="_grund",
        how="left"
    )
)

print(
    "✅ Bygning -> GJ jordstykke_id udfyldt:",
    int(bg["jordstykke_id"].notna().sum()),
    "af",
    len(bg),
    "rows"
)


# ============================================================
# STEP 3 - JOIN GRUNDJORDSTYKKE -> MATRIKEL
# ============================================================

print("\nSTEP 3) Join grundjordstykke -> MAT ...")

mat2 = mat.copy()

mat2["jordstykke_id"] = norm_join_key(mat2[MAT_KEY_COL])
mat2["ejerlavLokalId"] = norm_str(mat2["ejerlavLokalId"])
mat2["matrikelnummer_str"] = norm_str(mat2["matrikelnummer"])

mat2 = (
    mat2
    .dropna(subset=["jordstykke_id", "ejerlavLokalId", "matrikelnummer_str"])
    .drop_duplicates(["jordstykke_id", "ejerlavLokalId", "matrikelnummer_str"])
)

bgm = bg.merge(
    mat2[["jordstykke_id", "ejerlavLokalId", "matrikelnummer", "matrikelnummer_str"]],
    on="jordstykke_id",
    how="left"
)

match_rate_rows = float(bgm["matrikelnummer_str"].notna().mean())

match_rate_buildings = (
    bgm.groupby("building_id")["matrikelnummer_str"]
    .apply(lambda s: s.notna().any())
    .mean()
)

print("✅ GJ -> MAT match-rate rows:", round(match_rate_rows * 100, 2), "%")
print("✅ GJ -> MAT match-rate unique buildings:", round(match_rate_buildings * 100, 2), "%")


# ============================================================
# STEP 4 - CLEAN / VALID FILTER
# ============================================================

print("\nSTEP 4) Clean and filter valid rows ...")

required_valid = [
    "ejerlavLokalId",
    "matrikelnummer_str",
    "opfoer",
    "jordstykke_id",
    "building_id",
]

valid_raw = bgm.dropna(subset=required_valid).copy()

valid_raw = valid_raw[
    (valid_raw["opfoer"] >= MIN_PLAUSIBLE_YEAR)
    & (valid_raw["opfoer"] <= MAX_PLAUSIBLE_YEAR)
].copy()

before_dedup = len(valid_raw)

valid = valid_raw.drop_duplicates(
    subset=[
        "ejerlavLokalId",
        "matrikelnummer_str",
        "jordstykke_id",
        "building_id",
        "opfoer",
    ]
).copy()

removed_dupes = before_dedup - len(valid)

valid["matrikel_key"] = (
    valid["ejerlavLokalId"].astype("string")
    + "::"
    + valid["matrikelnummer_str"].astype("string")
)

print("✅ Valid rows før dedup:", before_dedup)
print("✅ Valid rows efter dedup:", len(valid))
print("✅ Fjernede eksakte dublet-rækker:", removed_dupes)
print("✅ Unikke bygninger i valid:", valid["building_id"].nunique())
print("✅ Unikke matrikler i valid:", valid["matrikel_key"].nunique())

if valid.empty:
    raise RuntimeError("valid er tom efter filtrering. Tjek join-nøgler og filterdefinition.")


# ============================================================
# STEP 5 - AUDIT: BYGNINGER PÅ FLERE MATRIKLER
# ============================================================

print("\nSTEP 5) Audit buildings mapped to multiple matrikler ...")

audit_building_matrikler = (
    valid.groupby("building_id", dropna=False)
    .agg(
        antal_rækker=("building_id", "size"),
        antal_jordstykker=("jordstykke_id", "nunique"),
        antal_matrikler=("matrikel_key", "nunique"),
        min_year=("opfoer", "min"),
        max_year=("opfoer", "max"),
        matrikler=("matrikel_key", uniq_join),
    )
    .reset_index()
    .sort_values(
        ["antal_matrikler", "antal_jordstykker", "antal_rækker"],
        ascending=False
    )
)

problem_buildings = audit_building_matrikler[
    audit_building_matrikler["antal_matrikler"] > 1
].copy()

problem_out = EXPORT / "audit_udlejning_beboelse_bygninger_paa_flere_matrikler.csv"
audit_all_out = EXPORT / "audit_udlejning_beboelse_alle_bygninger_matrikelkobling.csv"

problem_buildings.to_csv(problem_out, index=False, encoding="utf-8")
audit_building_matrikler.to_csv(audit_all_out, index=False, encoding="utf-8")

n_unique_buildings = valid["building_id"].nunique()
n_problem_buildings = problem_buildings["building_id"].nunique()

print(
    "⚠️ Bygninger koblet til flere matrikler:",
    n_problem_buildings,
    f"({pct(n_problem_buildings, n_unique_buildings):.2f}% af unikke bygninger)"
)

print("✅ Audit saved:", problem_out)
print("✅ Full audit saved:", audit_all_out)

if n_problem_buildings > 0:
    print("\nTop 10 bygninger koblet til flest matrikler:")
    display(problem_buildings.head(10))


# ============================================================
# STEP 6 - AGGREGATE PER MATRIKEL + CONSISTENT CUT-OFF
# ============================================================

print("\nSTEP 6) Aggregate per matrikel and classify ...")

grp = (
    valid.groupby(["ejerlavLokalId", "matrikelnummer_str"], dropna=False)
    .agg(
        min_year=("opfoer", "min"),
        max_year=("opfoer", "max"),
        unikke_bygninger=("building_id", "nunique"),
        unikke_jordstykker=("jordstykke_id", "nunique"),
        antal_udlejede_relevante_boligenheder=(
            "antal_udlejede_relevante_boligenheder",
            "sum"
        ),
        samlet_boligareal_enheder=("samlet_boligareal_enheder", "sum"),
        samlet_erhvervsareal_enheder=("samlet_erhvervsareal_enheder", "sum"),
    )
    .reset_index()
)

grp["matrikel_key"] = (
    grp["ejerlavLokalId"].astype("string")
    + "::"
    + grp["matrikelnummer_str"].astype("string")
)

grp["klasse"] = "ukendt"

grp.loc[
    grp["max_year"] <= OLD_MAX_YEAR,
    "klasse"
] = "kun_gammel"

grp.loc[
    grp["min_year"] >= NEW_MIN_YEAR,
    "klasse"
] = "kun_ny"

grp.loc[
    (grp["min_year"] <= OLD_MAX_YEAR)
    & (grp["max_year"] >= NEW_MIN_YEAR),
    "klasse"
] = "begge"

if (grp["klasse"] == "ukendt").any():
    bad = grp[grp["klasse"] == "ukendt"].head(20)
    raise RuntimeError(
        "Fandt matrikler der ikke passer i kun_gammel/kun_ny/begge. "
        "Eksempler:\n" + bad.to_string(index=False)
    )

klasse_counts = (
    grp["klasse"]
    .value_counts()
    .rename_axis("klasse")
    .reset_index(name="antal_matrikler")
)

both = (
    grp[grp["klasse"] == "begge"]
    .copy()
    .sort_values(["unikke_bygninger", "max_year"], ascending=[False, False])
)

print("✅ Antal matrikler i alt:", len(grp))
print("✅ Antal 'begge'-matrikler:", len(both))
print("\nMatrikler fordelt på klasse:")
display(klasse_counts)


# ============================================================
# STEP 7 - PUSH CLASS BACK TO BUILDING LEVEL
# ============================================================

print("\nSTEP 7) Push matrikel class back to building level ...")

v = valid.merge(
    grp[["ejerlavLokalId", "matrikelnummer_str", "klasse"]],
    on=["ejerlavLokalId", "matrikelnummer_str"],
    how="left",
    validate="m:1"
)

v["is_new_building"] = v["opfoer"] >= NEW_MIN_YEAR
v["is_old_building"] = v["opfoer"] <= OLD_MAX_YEAR

v["new_building_id"] = v["building_id"].where(v["is_new_building"])
v["old_building_id"] = v["building_id"].where(v["is_old_building"])

if v["klasse"].isna().any():
    raise RuntimeError("Nogle building-level rows fik ikke klasse efter merge.")


# ============================================================
# STEP 8 - COUNTS AND SMITTE-RISK SUMMARY
# ============================================================

print("\nSTEP 8) Count unique buildings and new-build distribution ...")

unikke_total = v["building_id"].nunique()
unikke_ny = v.loc[v["is_new_building"], "building_id"].nunique()
unikke_gammel = v.loc[v["is_old_building"], "building_id"].nunique()

nybyg_kun_ny = v.loc[
    (v["klasse"] == "kun_ny") & (v["is_new_building"]),
    "building_id"
].nunique()

nybyg_begge = v.loc[
    (v["klasse"] == "begge") & (v["is_new_building"]),
    "building_id"
].nunique()

nybyg_kun_gammel = v.loc[
    (v["klasse"] == "kun_gammel") & (v["is_new_building"]),
    "building_id"
].nunique()

buildings_by_class = (
    v.groupby("klasse", dropna=False)
    .agg(
        unikke_bygninger_total=("building_id", "nunique"),
        unikke_nybyg_bygninger=("new_building_id", "nunique"),
        unikke_gamle_bygninger=("old_building_id", "nunique"),
        antal_rækker=("building_id", "size"),
        unikke_matrikler=("matrikel_key", "nunique"),
        unikke_jordstykker=("jordstykke_id", "nunique"),
        antal_udlejede_relevante_boligenheder=(
            "antal_udlejede_relevante_boligenheder",
            "sum"
        ),
        samlet_boligareal_enheder=("samlet_boligareal_enheder", "sum"),
        samlet_erhvervsareal_enheder=("samlet_erhvervsareal_enheder", "sum"),
    )
    .reset_index()
    .sort_values("unikke_nybyg_bygninger", ascending=False)
)

summary = pd.DataFrame([
    {
        "metric": "unikke_bygninger_total_udlejning_beboelse",
        "value": unikke_total,
        "pct_af_nybyg": None,
    },
    {
        "metric": "unikke_gamle_bygninger_opfoer <= 1991",
        "value": unikke_gammel,
        "pct_af_nybyg": None,
    },
    {
        "metric": "unikke_nybyg_bygninger_opfoer >= 1992",
        "value": unikke_ny,
        "pct_af_nybyg": 100.0,
    },
    {
        "metric": "nybyg_paa_kun_ny_matrikler",
        "value": nybyg_kun_ny,
        "pct_af_nybyg": pct(nybyg_kun_ny, unikke_ny),
    },
    {
        "metric": "nybyg_paa_begge_matrikler_smitterisiko",
        "value": nybyg_begge,
        "pct_af_nybyg": pct(nybyg_begge, unikke_ny),
    },
    {
        "metric": "nybyg_paa_kun_gammel_matrikler_burde_vaere_0",
        "value": nybyg_kun_gammel,
        "pct_af_nybyg": pct(nybyg_kun_gammel, unikke_ny),
    },
    {
        "metric": "bygninger_koblet_til_flere_matrikler_audit",
        "value": n_problem_buildings,
        "pct_af_nybyg": None,
    },
])

print("\n=== CONSISTENT CUT-OFF FOR UDLEJNINGSEJENDOMME TIL BEBOELSE ===")
print(f"Old <= {OLD_MAX_YEAR}")
print(f"New >= {NEW_MIN_YEAR}")
print()
print("Unikke bygninger i valid total:", unikke_total)
print("Unikke gamle bygninger opført <= 1991:", unikke_gammel)
print("Unikke nybyg-bygninger opført >= 1992:", unikke_ny)
print()
print(
    "Nybyg på kun_ny matrikler:",
    nybyg_kun_ny,
    f"({pct(nybyg_kun_ny, unikke_ny):.2f}%)"
)
print(
    "Nybyg på begge matrikler, proxy for smitterisiko:",
    nybyg_begge,
    f"({pct(nybyg_begge, unikke_ny):.2f}%)"
)
print(
    "Nybyg på kun_gammel matrikler, bør være 0:",
    nybyg_kun_gammel,
    f"({pct(nybyg_kun_gammel, unikke_ny):.2f}%)"
)

print("\nBygninger pr. klasse:")
display(buildings_by_class)

print("\nSummary:")
display(summary)


# ============================================================
# STEP 9 - LEAK CHECK
# ============================================================

print("\nSTEP 9) Leak check ...")

leaks = v[
    (v["klasse"] == "kun_gammel")
    & (v["is_new_building"])
][
    [
        "ejerlavLokalId",
        "matrikelnummer_str",
        "jordstykke_id",
        "building_id",
        "opfoer",
        "klasse",
    ]
].drop_duplicates()

leaks_out = EXPORT / "audit_udlejning_beboelse_leaks_nybyg_paa_kun_gammel.csv"
leaks.to_csv(leaks_out, index=False, encoding="utf-8")

if len(leaks) > 0:
    print("⚠️ Fandt nybyg på kun_gammel matrikler. Tjek data.")
    display(leaks.head(20))
else:
    print("✅ Ingen nybyg på kun_gammel matrikler.")

print("✅ Leak audit saved:", leaks_out)


# ============================================================
# STEP 10 - EXPORT TABLES
# ============================================================

print("\nSTEP 10) Export tables ...")

out_both = EXPORT / f"udlejning_beboelse_matrikler_begge_cutoff_{CUT_YEAR}_years_{MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR}.csv"
out_grp = EXPORT / f"udlejning_beboelse_matrikler_alle_klasser_cutoff_{CUT_YEAR}_years_{MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR}.csv"
out_cls = EXPORT / f"udlejning_beboelse_klasse_fordeling_cutoff_{CUT_YEAR}_years_{MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR}.csv"
out_buildings_by_class = EXPORT / f"udlejning_beboelse_bygninger_pr_klasse_cutoff_{CUT_YEAR}.csv"
out_summary = EXPORT / f"udlejning_beboelse_summary_cutoff_{CUT_YEAR}.csv"
out_enh_summary = EXPORT / "udlejning_beboelse_enhed_summary_bygning.csv"

both.to_csv(out_both, index=False, encoding="utf-8")
grp.to_csv(out_grp, index=False, encoding="utf-8")
klasse_counts.to_csv(out_cls, index=False, encoding="utf-8")
buildings_by_class.to_csv(out_buildings_by_class, index=False, encoding="utf-8")
summary.to_csv(out_summary, index=False, encoding="utf-8")
enh_unit_summary.to_csv(out_enh_summary, index=False, encoding="utf-8")

if EXPORT_VALID_BUILDING_LEVEL:
    out_valid = EXPORT / f"udlejning_beboelse_building_level_valid_classified_cutoff_{CUT_YEAR}.csv"
    v.to_csv(out_valid, index=False, encoding="utf-8")
    print("✅ saved:", out_valid)

print("✅ saved:", out_both)
print("✅ saved:", out_grp)
print("✅ saved:", out_cls)
print("✅ saved:", out_buildings_by_class)
print("✅ saved:", out_summary)
print("✅ saved:", out_enh_summary)


# ============================================================
# STEP 11 - PLOTS
# ============================================================

print("\nSTEP 11) Create plots ...")

fig1 = EXPORT / "udlejning_beboelse_hist_min_year.png"
fig2 = EXPORT / "udlejning_beboelse_klasse_fordeling.png"
fig3 = EXPORT / "udlejning_beboelse_nybyg_bygninger_pr_klasse.png"

plt.figure()
grp["min_year"].dropna().astype(int).hist(bins=50)
plt.title(
    f"Fordeling af tidligste opførelsesår pr. matrikel "
    f"for udlejning/beboelse ({MIN_PLAUSIBLE_YEAR}-{MAX_PLAUSIBLE_YEAR})"
)
plt.xlabel("Min opførelsesår")
plt.ylabel("Antal matrikler")
plt.savefig(fig1, dpi=200, bbox_inches="tight")
plt.close()

plt.figure()
klasse_counts.set_index("klasse")["antal_matrikler"].plot(kind="bar")
plt.title(f"Matrikler fordelt på klasse, udlejning/beboelse, cutoff={CUT_YEAR}")
plt.xlabel("Klasse")
plt.ylabel("Antal matrikler")
plt.savefig(fig2, dpi=200, bbox_inches="tight")
plt.close()

plt.figure()
buildings_by_class.set_index("klasse")["unikke_nybyg_bygninger"].plot(kind="bar")
plt.title(f"Nybyg-bygninger fordelt på matrikelklasse, udlejning/beboelse, cutoff={CUT_YEAR}")
plt.xlabel("Klasse")
plt.ylabel("Unikke nybyg-bygninger")
plt.savefig(fig3, dpi=200, bbox_inches="tight")
plt.close()

print("✅ saved figs:", fig1, fig2, fig3)


# ============================================================
# STEP 12 - FINAL OUTPUT
# ============================================================

print("\n=== DONE ===")
print("Analysepopulation:")
print("- Bygninger med byg021BygningensAnvendelse i:", BUILDING_RESIDENTIAL_CODES)
print("- Enheder med enh020EnhedensAnvendelse i:", UNIT_RESIDENTIAL_CODES)
print("- Enheder med enh045Udlejningsforhold i:", RENTED_CODES)
print("- Minimum udlejede relevante boligenheder:", MIN_RENTED_RESIDENTIAL_UNITS)
print()
print("Vigtigste output-objekter i notebook:")
print("- byg: filtrerede bygninger, dvs. analysepopulationen")
print("- enh: enhedsdata med flags")
print("- enh_unit_summary: enheder aggregeret til bygning")
print("- valid: renset building/matrikel-level datasæt før klasse-merge")
print("- v: building/matrikel-level datasæt med klasse og ny/gammel flags")
print("- grp: matrikel-level datasæt med min_year, max_year og klasse")
print("- both: matrikler med både gamle og nye bygninger")
print("- buildings_by_class: bygningstællinger pr. klasse")
print("- problem_buildings: bygninger koblet til flere matrikler")
print("- summary: samlet hovedresultat")

both.head(10)

STEP 0) Find inputfiler ...
✅ Bygning: data/BBR_V3_Bygning_0101_TotalDownload_csv_Current_535.csv
✅ GrundJordstykke: data/BBR_V3_GrundJordstykke_0101_TotalDownload_csv_Current_535.csv
✅ Enhed: data/BBR_V3_Enhed_0101_TotalDownload_csv_Current_521.csv
✅ MAT Jordstykke: data/mat_jordstykke/MAT_V3_Jordstykke_TotalDownload_csv_Temporal_574.csv
✅ Provenance saved: exports/input_provenance.csv

STEP 1) Load CSV-filer ...
✅ Loaded byg: 92284 rows | 5 cols
✅ Loaded gj : 38725 rows | 19 cols
✅ Loaded enh: 488767 rows | 10 cols
✅ Loaded mat: 3336383 rows | 3 cols

STEP 1A) Kommune-filter ...
✅ Byg kommune-filter 0101: 92281 rows tilbage af 92284
✅ Enhed kommune-filter 0101: 488767 rows tilbage af 488767

STEP 1B) Filtrér til udlejningsejendomme til beboelse ...

Fordeling i Bygning - byg021BygningensAnvendelse:


,byg021,antal_bygninger
0,930,22407
1,120,15791
2,140,13591
3,920,6732
4,910,6483
5,540,4822
6,131,4791
7,321,2392
8,132,1863
9,950,1652



Fordeling i Enhed - enh020EnhedensAnvendelse:


,enh020,antal_enheder
0,140,335671
1,<NA>,63969
2,120,16223
3,321,10218
4,150,9964
5,131,7989
6,322,6893
7,590,6687
8,540,4432
9,333,3444



Fordeling i Enhed - enh045Udlejningsforhold:


...

,klasse,unikke_bygninger_total,unikke_nybyg_bygninger,unikke_gamle_bygninger,antal_rækker,unikke_matrikler,unikke_jordstykker,antal_udlejede_relevante_boligenheder,samlet_boligareal_enheder,samlet_erhvervsareal_enheder
2,kun_ny,989,989,0,1076,686,675,39685,4943313.0,890096.0
0,begge,104,35,69,104,16,16,1031,97968.0,18206.0
1,kun_gammel,9212,0,9212,9342,7787,7762,206329,19071084.0,2026654.0



Summary:


,metric,value,pct_af_nybyg
0,unikke_bygninger_total_udlejning_beboelse,10305,NaN
1,unikke_gamle_bygninger_opfoer <= 1991,9281,NaN
2,unikke_nybyg_bygninger_opfoer >= 1992,1024,100.000000
3,nybyg_paa_kun_ny_matrikler,989,96.582031
4,nybyg_paa_begge_matrikler_smitterisiko,35,3.417969
5,nybyg_paa_kun_gammel_matrikler_burde_vaere_0,0,0.000000
6,bygninger_koblet_til_flere_matrikler_audit,168,NaN



STEP 9) Leak check ...
✅ Ingen nybyg på kun_gammel matrikler.
✅ Leak audit saved: exports/audit_udlejning_beboelse_leaks_nybyg_paa_kun_gammel.csv

STEP 10) Export tables ...
✅ saved: exports/udlejning_beboelse_matrikler_begge_cutoff_1992_years_1500-2026.csv
✅ saved: exports/udlejning_beboelse_matrikler_alle_klasser_cutoff_1992_years_1500-2026.csv
✅ saved: exports/udlejning_beboelse_klasse_fordeling_cutoff_1992_years_1500-2026.csv
✅ saved: exports/udlejning_beboelse_bygninger_pr_klasse_cutoff_1992.csv
✅ saved: exports/udlejning_beboelse_summary_cutoff_1992.csv
✅ saved: exports/udlejning_beboelse_enhed_summary_bygning.csv

STEP 11) Create plots ...
✅ saved figs: exports/udlejning_beboelse_hist_min_year.png exports/udlejning_beboelse_klasse_fordeling.png exports/udlejning_beboelse_nybyg_bygninger_pr_klasse.png

=== DONE ===
Analysepopulation:
- Bygninger med byg021BygningensAnvendelse i: {140}
- Enheder med enh020EnhedensAnvendelse i: {140}
- Enheder med enh045Udlejningsforhold i: {1}
- 

,ejerlavLokalId,matrikelnummer_str,min_year,max_year,unikke_bygninger,unikke_jordstykker,antal_udlejede_relevante_boligenheder,samlet_boligareal_enheder,samlet_erhvervsareal_enheder,matrikel_key,klasse
2164,2000170,29,1967,2001,50,1,282,8517.0,77.0,2000170::29,begge
2372,2000171,2290,1974,2020,15,1,203,22604.0,3330.0,2000171::2290,begge
6208,2000174,265,1988,1992,7,1,175,13365.0,0.0,2000174::265,begge
3002,2000172,4077,1950,2020,4,1,81,7559.0,6218.0,2000172::4077,begge
601,2000153,571,1870,1998,4,1,14,1707.0,388.0,2000153::571,begge
5854,2000174,119,1854,2023,3,1,30,9806.0,2648.0,2000174::119,begge
1825,2000166,51,1860,2016,3,1,15,10114.0,-2416.0,2000166::51,begge
2230,2000171,13e,1883,2021,2,1,28,2864.0,3016.0,2000171::13e,begge
5043,2000173,4821,1918,2020,2,1,16,1737.0,1967.0,2000173::4821,begge
5482,2000173,600,1920,2018,2,1,11,4565.0,2067.0,2000173::600,begge


In [3]:
# Alle bygninger på 'begge'-matrikler
begge_alle = v[v["klasse"] == "begge"].drop_duplicates(subset=["building_id"])
print("Antal unikke bygninger i alt:", begge_alle["building_id"].nunique())
print("Heraf nybyg (>= 1992):", begge_alle[begge_alle["is_new_building"]]["building_id"].nunique())
print("Heraf gammel (<= 1991):", begge_alle[begge_alle["is_old_building"]]["building_id"].nunique())

# Areal
print("\nSamlet boligareal (m²):", begge_alle["samlet_boligareal_enheder"].sum())
print("Samlet erhvervsareal (m²):", begge_alle["samlet_erhvervsareal_enheder"].sum())

Antal unikke bygninger i alt: 104
Heraf nybyg (>= 1992): 35
Heraf gammel (<= 1991): 69

Samlet boligareal (m²): 97968.0
Samlet erhvervsareal (m²): 18206.0
